# 🧠 Intersection over Union (IoU): การทับซ้อนทางคณิตศาสตร์และตัวชี้วัดการสูญเสีย (Loss Metrics)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Intersection over Union (IoU)**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายทฤษฎี, การอนุพัทธ์สูตรทีละขั้นตอน และความสำคัญของ IoU
2. พัฒนาการคำนวณ IoU จากศูนย์ด้วย NumPy
3. กำหนดกรอบขอบเขต (bounding boxes) ที่มีระดับการทับซ้อนที่แตกต่างกันดังนี้:
   - การทับซ้อนสูง (High Overlap)
   - การทับซ้อนต่ำ (Low Overlap)
   - การไม่ทับซ้อนกันเลย (Zero Overlap)
4. คำนวณและเปรียบเทียบค่า IoU สำหรับแต่ละกรณี
5. พล็อตกราฟเปรียบเทียบกรอบขอบเขตเคียงข้างกันโดยใช้ Matplotlib เพื่อแสดงภาพพื้นที่ส่วนที่ซ้อนทับกันอย่างชัดเจน
6. อภิปรายการคำนวณการสูญเสียของ IoU ขั้นสูง (GIoU, DIoU, CIoU) ที่ใช้ในโมเดลการตรวจจับวัตถุยุคปัจจุบัน (YOLO)

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set seed for reproducibility
np.random.seed(42)

## 1. การคำนวณ IoU จากศูนย์ด้วย NumPy

เราจะเขียนฟังก์ชันที่รับค่าพิกัดของกรอบขอบเขตสองกรอบในรูปแบบ XYXY และคำนวณอัตราส่วนการทับซ้อนออกมาเป็นตัวเลขทศนิยมที่แม่นยำครับ

In [ ]:
def calculate_iou(boxA, boxB):
    x1_inter = max(boxA[0], boxB[0])
    y1_inter = max(boxA[1], boxB[1])
    x2_inter = min(boxA[2], boxB[2])
    y2_inter = min(boxA[3], boxB[3])
    
    width_inter = max(0.0, x2_inter - x1_inter)
    height_inter = max(0.0, y2_inter - y1_inter)
    area_inter = width_inter * height_inter
    
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    area_union = areaA + areaB - area_inter
    
    if area_union == 0.0:
        return 0.0
        
    return area_inter / area_union

## 2. การกำหนดกรณีศึกษาของกรอบขอบเขต

เรากำหนดกรอบขอบเขตสี่กรอบในรูปแบบ XYXY ดังนี้:
-   **กรอบจริงอ้างอิงเป้าหมาย A (Ground Truth Box A):** `[100, 100, 200, 200]`
-   **กรอบ B (การทับซ้อนสูง):** `[120, 120, 220, 220]`
-   **กรอบ C (การทับซ้อนต่ำ):** `[180, 180, 280, 280]`
-   **กรอบ D (การไม่ทับซ้อนกันเลย):** `[250, 250, 350, 350]`

In [ ]:
box_truth = [100.0, 100.0, 200.0, 200.0]
box_high = [120.0, 120.0, 220.0, 220.0]
box_low = [180.0, 180.0, 280.0, 280.0]
box_zero = [250.0, 250.0, 350.0, 350.0]

iou_high = calculate_iou(box_truth, box_high)
iou_low = calculate_iou(box_truth, box_low)
iou_zero = calculate_iou(box_truth, box_zero)

print(f"High Overlap IoU: {iou_high:.4f}")
print(f"Low Overlap IoU : {iou_low:.4f}")
print(f"Zero Overlap IoU: {iou_zero:.4f}")

## 3. การแสดงผลภาพการทับซ้อนของกรอบขอบเขต

เรามาพล็อตกราฟทั้งสามกรณีศึกษาข้างกันโดยใช้ Matplotlib เพื่อให้เห็นภาพการเปลี่ยนแปลงของพื้นที่ส่วนที่ทับซ้อนกันครับ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

boxes = [box_high, box_low, box_zero]
titles = [f"High Overlap (IoU = {iou_high:.4f})", f"Low Overlap (IoU = {iou_low:.4f})", f"Zero Overlap (IoU = {iou_zero:.4f})"]

for idx, ax in enumerate(axes):
    ax.set_xlim(50, 400)
    ax.set_ylim(400, 50)
    
    rect_truth = patches.Rectangle((box_truth[0], box_truth[1]), 100, 100, linewidth=3, edgecolor='green', facecolor='none', label='Ground Truth')
    ax.add_patch(rect_truth)
    
    p_box = boxes[idx]
    w = p_box[2] - p_box[0]
    h = p_box[3] - p_box[1]
    rect_pred = patches.Rectangle((p_box[0], p_box[1]), w, h, linewidth=3, edgecolor='red', facecolor='none', label='Prediction')
    ax.add_patch(rect_pred)
    
    x1_i = max(box_truth[0], p_box[0])
    y1_i = max(box_truth[1], p_box[1])
    x2_i = min(box_truth[2], p_box[2])
    y2_i = min(box_truth[3], p_box[3])
    
    w_i = max(0.0, x2_i - x1_i)
    h_i = max(0.0, y2_i - y1_i)
    if w_i > 0 and h_i > 0:
        rect_inter = patches.Rectangle((x1_i, y1_i), w_i, h_i, facecolor='blue', alpha=0.3, label='Intersection')
        ax.add_patch(rect_inter)
        
    ax.set_title(titles[idx], fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

## 4. ทำไม YOLO ถึงใช้ค่าสูญเสียแบบ CIoU (CIoU Loss)

การคำนวณ IoU มาตรฐานมีข้อจำกัดหลักสองประการ:
1.  **ความอิ่มตัวของเกรเดียนต์ (Gradient Saturation):** เมื่อกรอบไม่มีส่วนทับซ้อนกันเลย ค่า $\text{IoU} = 0$ ส่งผลให้เกรเดียนต์ของฟังก์ชันการสูญเสียมีค่าเป็น 0 ทำให้โครงข่ายประสาทเทียมไม่สามารถเรียนรู้วิธีการปรับเลื่อนกรอบให้ขยับเข้าใกล้กันได้
2.  **ขาดการจัดตำแหน่งรูปทรง (Lack of Shape Alignment):** การหา IoU ปกติจะไม่นำเรื่องของอัตราส่วนกว้างยาว (aspect ratios) ของกรอบวัตถุมาพิจารณาว่าตรงกันดีเพียงใด

เพื่อแก้ไขปัญหานี้ YOLO จึงใช้ **Complete IoU (CIoU) Loss**:
$$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(\mathbf{b}, \mathbf{b}^{gt})}{c^2} + \alpha v$$

โดยที่:
-   $\rho(\mathbf{b}, \mathbf{b}^{gt})$ คือระยะทางยูคลิเดียนระหว่างจุดกึ่งกลางของกรอบที่ทำนายและกรอบเป้าหมายจริง
-   $c$ คือความยาวเส้นทแยงมุมของกรอบปิดล้อมที่เล็กที่สุดที่ครอบคลุมกรอบทั้งสองเอาไว้
-   $\alpha v$ คือพจน์ความสอดคล้องของอัตราส่วนกว้างยาว (aspect ratio consistency term) ที่วัดว่าสัดส่วนความกว้างและความสูงของกรอบทำนายตรงกับกรอบจริงเพียงใด
สิ่งนี้ช่วยรับประกันว่าจะมีการปรับปรุงตำแหน่งของกรอบได้อย่างมั่นคงเสถียร แม้ในกรณีที่กรอบตรวจจับไม่ทับซ้อนกับกรอบจริงเลยก็ตาม!